In [1]:
from pyvis.network import Network
import pickle
import os
import json
import numpy as np

In [2]:
result_path = "/nfs/data/fakenarratives/202306_corpus/results_pkl/Tagesschau/TV-20230120-2022-1800.webl.h264"

## Read pickles
shots_dict = pickle.load(open(os.path.join(result_path, "transnet_shotdetection.pkl"), "rb"))
asr_dict = pickle.load(open(os.path.join(result_path, "asr.pkl"), "rb"))
nlp_dict = pickle.load(open(os.path.join(result_path, "asr_textnlp.pkl"), "rb"))
event_dict = pickle.load(open(os.path.join(result_path, "eventclassification_vise.pkl"), "rb"))
place_dict = pickle.load(open(os.path.join(result_path, "place_classification.pkl"), "rb"))

In [3]:
print(shots_dict.keys())
print(event_dict.keys())
print(place_dict.keys())
print(asr_dict.keys())
print(nlp_dict.keys())

dict_keys(['inputs', 'outputs', 'pipeline', 'plugins', 'video_file', 'video_id', 'output_data'])
dict_keys(['leaf_node_vector', 'leaf_node_labels', 'time', 'delta_time', 'config', 'args'])
dict_keys(['inputs', 'outputs', 'pipeline', 'plugins', 'video_file', 'video_id', 'output_data'])
dict_keys(['github_repo', 'commit_id', 'parameters', 'video_file', 'output_data'])
dict_keys(['github_repo', 'commit_id', 'parameters', 'video_file', 'output_data'])


In [4]:
print(shots_dict["output_data"].keys())
print(asr_dict["output_data"].keys())

dict_keys(['id', 'version', 'type', 'name', 'ref_id', 'shots'])
dict_keys(['text', 'segments', 'language', 'speaker_segments'])


In [15]:
event_index = event_dict["leaf_node_labels"]
len(event_dict["leaf_node_vector"])

4544

In [20]:
# Create tuples of (time, event), (time, place16)
event_index = event_dict["leaf_node_labels"]
event_list = []
for pred_vector, t in zip(event_dict["leaf_node_vector"], event_dict["time"]):
    print(pred_vector.shape)
    print(np.argmax(pred_vector))
    label = event_index[np.argmax(pred_vector)]
    print(label, t)
    event_list.append((label, t))
    
    
place_index = place_dict["output_data"]["probs_places16"]["index"]
place_list = []
for pred_vector, t in zip(place_dict["output_data"]["probs_places16"]["y"], place_dict["output_data"]["probs_places16"]["time"]):
    label = place_index[np.argmax(pred_vector)]
    place_list.append((label, t))

(148,)
38
cyclone/tornado 0.0
(148,)
38
cyclone/tornado 0.2
(148,)
48
esports 0.4
(148,)
48
esports 0.6
(148,)
48
esports 0.8
(148,)
48
esports 1.0
(148,)
48
esports 1.2
(148,)
48
esports 1.4
(148,)
48
esports 1.6
(148,)
48
esports 1.8
(148,)
48
esports 2.0
(148,)
48
esports 2.2
(148,)
48
esports 2.4
(148,)
48
esports 2.6
(148,)
48
esports 2.8
(148,)
48
esports 3.0
(148,)
48
esports 3.2
(148,)
48
esports 3.4
(148,)
48
esports 3.6
(148,)
48
esports 3.8
(148,)
4
archery 4.0
(148,)
48
esports 4.2
(148,)
45
election/referendum/political campaign 4.4
(148,)
45
election/referendum/political campaign 4.6
(148,)
45
election/referendum/political campaign 4.8
(148,)
45
election/referendum/political campaign 5.0
(148,)
45
election/referendum/political campaign 5.2
(148,)
45
election/referendum/political campaign 5.4
(148,)
45
election/referendum/political campaign 5.6
(148,)
45
election/referendum/political campaign 5.8
(148,)
45
election/referendum/political campaign 6.0
(148,)
45
election/refer

In [6]:
def add_shot_nodes(net, shots):
    for i, shot in enumerate(shots):
        start_time = shot["start"]
        end_time = shot["end"]

        label = f"Shot {i}"
        title = f"Start: {start_time}, End: {end_time}"

        net.add_node(f"shot_{i}", label=label, title=title, 
                     start_time=start_time, end_time=end_time, color="#ff9999", shape="box", level=1)
    
    # Add edges within shot nodes and segment nodes
    for i in range(len(shots) - 1):
        net.add_edge(f"shot_{i}", f"shot_{i+1}")
    
    return net

In [7]:
def add_speaker_nodes(net, speaker_segments):
    for i, segment in enumerate(speaker_segments):
        start_time = round(segment["start"],2)
        end_time = round(segment["end"],2)
        speaker_label = segment["speaker"]

        # Calculate the number of words in the segment (as one of the attributes just for check)
        segment_text = segment["text"]
        num_words = len(segment_text.split())

        # Create the label and title for the node (To see the attribute values and label nodes)
        label = f"Spk. segment {i}"
        title = f"Speaker ID: {speaker_label}, Num Words: {num_words}, Start: {start_time}, End: {end_time}"

        net.add_node(f"spk_segment_{i}", label=label, title=title, 
                     start_time=start_time, end_time=end_time, color="#69cfd3", shape="box", level=3)
    
    for i in range(len(speaker_segments) - 1):
        net.add_edge(f"spk_segment_{i}", f"spk_segment_{i+1}")
        
    return net

In [8]:
def add_span_nodes(net, speaker_segments, shots, event_list, place_list):
    spans = []
    current_start = 0

    # Iterate over the speaker segments
    for segment in speaker_segments:
        # If the span is before a speaker segment or inbetween speaker segments
        if segment['start'] > current_start:
            spans.append({'start': current_start, 'end': round(segment['start'], 2)})
            current_start = round(segment['start'],2)

        # If the span is part of the speaker segment
        if segment['end'] > current_start:
            spans.append({'start': current_start, 'end': round(segment['end'], 2)})
            current_start = round(segment['end'], 2)

    # For the last remaining span after last speaker segment (Take from last shot)
    video_end = round(shots[-1]['end'], 2)
    spans.append({'start': current_start, 'end': video_end})

    for i, span in enumerate(spans):
        start_time = span["start"]
        end_time = span["end"]
        
        # Events in the span
        span_events = list(set([e for e, t in event_list if t >= start_time and t <= end_time]))
        
        # Places in the span
        span_places = list(set([e for e, t in place_list if t >= start_time and t <= end_time]))

        label = f"Span {i}"
        title = f"Start: {start_time}, End: {end_time}, \
                Events: {', '.join(span_events)}, Places: {', '.join(span_places)}"

        net.add_node(f"span_{i}", label=label, title=title, events=span_events, places=span_places,
                     start_time=start_time, end_time=end_time, color="#66cc66", level=2)

    for i in range(len(spans) - 1):
        net.add_edge(f"span_{i}", f"span_{i+1}")
        
    return net, spans

In [9]:
def add_edges_to_spans(net, shots, speaker_segments, spans):
    # Iterate over the shots
    for i, shot in enumerate(shots):
        start_time = shot["start"]
        end_time = shot["end"]
        
        # Find the spans that overlap with this shot
        overlapping_spans = [f"span_{j}" for j, span in enumerate(spans) 
                             if not (end_time <= span["start"] or start_time >= span["end"])]

        # Add edges from the shot to the overlapping spans
        for span_id in overlapping_spans:
            net.add_edge(f"shot_{i}", span_id)

    # Iterate over the speaker segments
    for i, segment in enumerate(speaker_segments):
        start_time = segment["start"]
        end_time = segment["end"]
        
        # Find the spans that overlap with this speaker segment
        overlapping_spans = [f"span_{j}" for j, span in enumerate(spans) 
                             if not (end_time <= span["start"] or start_time >= span["end"])]

        # Add edges from the speaker segment to the overlapping spans
        for span_id in overlapping_spans:
            net.add_edge(f"spk_segment_{i}", span_id)
            
    return net

In [25]:
# Create a PyVis network
net = Network(notebook=True, directed=True)

options = {
    "layout": {
        "hierarchical": {
            "direction": "LR"  # Left to right
        }
    }
}

net.set_options(json.dumps(options))

# Limiting to 10 shots
shots = shots_dict["output_data"]["shots"][:10]

# limite to 10 speaker segments
speaker_segments = asr_dict["output_data"]["speaker_segments"][:10]

net = add_shot_nodes(net, shots)

# Add spans (Smallest temporal element in our graph)
net, spans = add_span_nodes(net, speaker_segments, shots, event_list, place_list)

print(len(spans))

net = add_speaker_nodes(net, speaker_segments)

net = add_edges_to_spans(net, shots, speaker_segments, spans)

net.show("graphs/segment_net.html")

20
graphs/segment_net.html
